In [1]:
import os
import shutil
from datetime import datetime, timezone
from PIL import Image
import numpy as np
import pandas as pd

def datetime_utc(year, month, day, hour=0, minute=0, second=0):
    return datetime(year, month, day, hour, minute, second, tzinfo=timezone.utc)


def parse_timestamp_from_filename(fname):
    parts = fname.split(".")
    try:
        return datetime.fromtimestamp(int(parts[0]), tz=timezone.utc)
    except (ValueError, IndexError):
        pass
    try:
        day, hour, minute, second = parts[3].split("_")
        dt_str = f"{day} {parts[2]} {parts[5]} {hour}:{minute}:{second}"
        return datetime.strptime(dt_str, "%d %b %Y %H:%M:%S").replace(tzinfo=timezone.utc)
    except (ValueError, IndexError):
        return None


def select_files_by_date(folder_path, start_dt, end_dt, extensions=(".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")):
    matched = []
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith(extensions):
            continue
        dt = parse_timestamp_from_filename(fname)
      
        if dt is None:
            print(f"  [SKIP] Could not parse timestamp: {fname}")
            continue
        if start_dt <= dt <= end_dt:
            matched.append(fname)
    return matched


def is_all_black(image_path, threshold=5):
    try:
        with Image.open(image_path) as img:
            arr = np.array(img.convert("L"))
            return arr.mean() <= threshold
    except Exception as e:
        print(f"  [ERROR] Reading {image_path}: {e}")
        return False


def quarantine_black_images(folder_path, threshold=5, dry_run=True, quarantine_folder=None):
    if quarantine_folder is None:
        quarantine_folder = os.path.join(folder_path, "quarantine_black")
    quarantined = []
    for fname in os.listdir(folder_path):
        if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")):
            continue
        path = os.path.join(folder_path, fname)
        if is_all_black(path, threshold=threshold):
            dest = os.path.join(quarantine_folder, fname)
            if dry_run:
                print(f"  [DRY RUN] Would quarantine: {fname}")
            else:
                os.makedirs(quarantine_folder, exist_ok=True)
                shutil.move(path, dest)
                print(f"  Quarantined: {fname}")
                quarantined.append(fname)
    return quarantined

In [ ]:
quarantine_black_images('E:/cvolpano/DATA/CoastCam/MarconiTimestacks/01-14_02-02_selected',threshold=40, dry_run=False, quarantine_folder='E:/cvolpano/DATA/CoastCam/MarconiTimestacks/01-14_02-02_selected/quarantine_black')

In [2]:

def run_pipeline(source_folder, start_dt, end_dt, output_folder,
                 threshold=5, dry_run=True, quarantine_folder=None):
    """
    Full pipeline:
      1. Select files in [start_dt, end_dt] from source_folder
      2. Copy them into output_folder
      3. Scan output_folder for all-black images
      4. Move black images to quarantine_folder

    dry_run=True  → previews every step, no files are moved or copied.
    dry_run=False → actually runs all operations.
    """

    print("=" * 60)
    print(f"SOURCE:      {source_folder}")
    print(f"OUTPUT:      {output_folder}")
    print(f"DATE RANGE:  {start_dt}  →  {end_dt}")
    print(f"THRESHOLD:   {threshold}")
    print(f"DRY RUN:     {dry_run}")
    print("=" * 60)

    # ------------------------------------------------------------------
    # Step 1: Select files by date range
    # ------------------------------------------------------------------
    print("\n[STEP 1] Selecting files by date range...")
    selected = select_files_by_date(source_folder, start_dt, end_dt)
    print(f"  → {len(selected)} file(s) matched.\n")

    if not selected:
        print("No files matched the date range. Exiting.")
        return

    # ------------------------------------------------------------------
    # Step 2: Copy selected files to output folder
    # ------------------------------------------------------------------
    print("[STEP 2] Copying selected files to output folder...")
    copied = []
    for fname in selected:
        src  = os.path.join(source_folder, fname)
        dest = os.path.join(output_folder, fname)
        if dry_run:
            print(f"  [DRY RUN] Would copy: {fname}")
        else:
            os.makedirs(output_folder, exist_ok=True)
            shutil.copy2(src, dest)
            print(f"  Copied: {fname}")
            copied.append(fname)

    count = len(selected) if dry_run else len(copied)
    print(f"  → {count} file(s) copied.\n")

    # ------------------------------------------------------------------
    # Step 3 + 4: Scan output folder and quarantine black images
    # ------------------------------------------------------------------
    print("[STEP 3+4] Scanning for black images and quarantining...")
    quarantined = quarantine_black_images(
        folder_path=output_folder,
        threshold=threshold,
        dry_run=dry_run,
        quarantine_folder=quarantine_folder
    )

    count = len(selected) if dry_run else len(quarantined)
    print(f"  → {count} black image(s) quarantined.\n")

       # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    print("=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"  Files matched date range : {len(selected)}")
    print(f"  Files copied             : {len(selected) if dry_run else len(copied)}")
    print(f"  Black images quarantined : {len(selected) if dry_run else len(quarantined)}")
    if not dry_run:
        clean = len(copied) - len(quarantined)
        print(f"  Clean images remaining   : {clean}")
    print("=" * 60)


In [5]:
df = pd.read_csv("C:/Users/cvolpano/PUVprocessing_usgs/nanopod/data/wave_data/cumulative_power_thresholds_2024-10-01-2025-03-31.csv")

# --- Parse datetimes ---
df["survey_date"]      = pd.to_datetime(df["survey_date"])
df["datetime_crossed"] = pd.to_datetime(df["datetime_crossed"])
print(df["survey_date"])

# --- Filter to only rows that surpass the 600 kWh/m threshold ---
TARGET = 600
exceeded = df[df["threshold_kWh_m"] >= TARGET].copy()

# --- Separate forward and backward ---
fwd = exceeded[exceeded["direction"] == "forward"].copy()
bwd = exceeded[exceeded["direction"] == "backward"].copy()

# --- For each survey_date + direction, keep the closest crossing ---
# Forward: smallest days_to_threshold (closest event ahead)
# Backward: smallest days_to_threshold (closest event behind)
fwd = fwd.sort_values("days_to_threshold").groupby("survey_date", as_index=False).first()
bwd = bwd.sort_values("days_to_threshold").groupby("survey_date", as_index=False).first()

combined = np.column_stack((
    bwd["datetime_crossed"].values,
    fwd["datetime_crossed"].values,
    
))
result = pd.DataFrame(combined, columns=["bkw_date", "fwd_date"])

print(result)

0    2024-10-23
1    2024-10-23
2    2024-10-23
3    2024-10-23
4    2024-10-23
5    2024-10-23
6    2025-01-23
7    2025-01-23
8    2025-01-23
9    2025-01-23
10   2025-01-23
11   2025-01-23
12   2025-02-19
13   2025-02-19
14   2025-02-19
15   2025-02-19
16   2025-02-19
17   2025-02-19
Name: survey_date, dtype: datetime64[us]
             bkw_date            fwd_date
0 2024-10-14 22:00:00 2024-11-01 14:00:00
1 2025-01-14 04:00:00 2025-02-02 02:00:00
2 2025-02-16 20:00:00 2025-03-04 23:00:00


In [6]:
import pandas as pd

source = r"E:/cvolpano/DATA/CoastCam/MarconiTimestacks"

# Ensure datetime columns are parsed correctly
result["fwd_date"] = pd.to_datetime(result["fwd_date"])
result["bkw_date"] = pd.to_datetime(result["bkw_date"])
row = result.iloc[2]
# for _, row in result.iterrows():

start = row["bkw_date"]   # backward crossing = earlier bound
end   = row["fwd_date"]   # forward crossing  = later bound

# Skip if either date is missing
if pd.isna(start) or pd.isna(end):
    print(f"  [SKIP] Missing date for survey")


# Convert to timezone-aware UTC for run_pipeline compatibility
start = start.tz_localize("UTC")
end   = end.tz_localize("UTC")

outfolder  = f"{start.strftime('%m-%d')}_{end.strftime('%m-%d')}"
output     = f"{source}/{outfolder}_selected"
quarantine = f"{output}/quarantine_black"

print(f"\n {start.date()} → {end.date()}")

run_pipeline(source, start, end, output,
                threshold=35, dry_run=False, quarantine_folder=quarantine)


 2025-02-16 → 2025-03-04
SOURCE:      E:/cvolpano/DATA/CoastCam/MarconiTimestacks
OUTPUT:      E:/cvolpano/DATA/CoastCam/MarconiTimestacks/02-16_03-04_selected
DATE RANGE:  2025-02-16 20:00:00+00:00  →  2025-03-04 23:00:00+00:00
THRESHOLD:   35
DRY RUN:     False

[STEP 1] Selecting files by date range...
  → 744 file(s) matched.

[STEP 2] Copying selected files to output folder...
  Copied: 1739736000.Sun.Feb.16_20_00_00.GMT.2025.CACO04.c1.ras.tiff
  Copied: 1739736000.Sun.Feb.16_20_00_00.GMT.2025.CACO04.c2.ras.tiff
  Copied: 1739737800.Sun.Feb.16_20_30_00.GMT.2025.CACO04.c1.ras.tiff
  Copied: 1739737800.Sun.Feb.16_20_30_00.GMT.2025.CACO04.c2.ras.tiff
  Copied: 1739739600.Sun.Feb.16_21_00_00.GMT.2025.CACO04.c1.ras.tiff
  Copied: 1739739600.Sun.Feb.16_21_00_00.GMT.2025.CACO04.c2.ras.tiff
  Copied: 1739741400.Sun.Feb.16_21_30_00.GMT.2025.CACO04.c1.ras.tiff
  Copied: 1739741400.Sun.Feb.16_21_30_00.GMT.2025.CACO04.c2.ras.tiff
  Copied: 1739743200.Sun.Feb.16_22_00_00.GMT.2025.CACO04.c1.ra